# Charts

Seven short lessons, each on one real dataset from [The Pudding](https://pudding.cool), taken
straight from [their public data repo](https://github.com/the-pudding/data).

1. A bar, for comparing two things
2. Bars hide the spread
3. A line, for change over time
4. Divide by whatever grew anyway
5. When the colour is the data
6. The category is a choice
7. The title is the finding

Then five harder ones, built out of the same parts:

8. Small multiples
9. A bump chart
10. A dumbbell chart
11. A heatmap
12. A scatter that says no

`pandas` holds the numbers, `matplotlib` draws them, `seaborn` sets the house style and
handles one chart that is fiddly by hand. All three are already on Colab.

In [ ]:
import io
import pandas as pd
import matplotlib.pyplot as plt
import requests
import seaborn as sns                    # already installed on Colab
from matplotlib.colors import LinearSegmentedColormap

PUDDING = "https://raw.githubusercontent.com/the-pudding/data/master/"

def pudding(path):
    """Read one of The Pudding's CSVs straight off GitHub."""
    return pd.read_csv(io.StringIO(requests.get(PUDDING + path, timeout=60).text))

One house style, set once. Two colours, since no chart here shows more than two things at a
time. A grey for anything that is context rather than data. One pale-to-dark ramp for the
heatmap.

In [ ]:
PAPER, INK, MUTED, RULE = "#FBFAF7", "#1A1A1A", "#5A5A52", "#E3E0D8"
RUST, TEAL, GREY = "#A8452B", "#0E6B95", "#B9B3A9"
ROAST = LinearSegmentedColormap.from_list(
    "roast", ["#F7EFE7", "#E2BFA4", "#CE9874", "#B5724C", "#98522F", "#77391F", "#522410"])

sns.set_theme(style="ticks")
plt.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 120,
    "figure.facecolor": PAPER, "axes.facecolor": PAPER, "savefig.facecolor": PAPER,
    "font.family": "sans-serif", "font.size": 9.5,
    "text.color": INK, "axes.labelcolor": MUTED, "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.edgecolor": RULE, "axes.linewidth": 0.8,
    "axes.titlelocation": "left", "axes.titlesize": 11, "axes.titleweight": "medium",
    "axes.titlecolor": INK, "axes.titlepad": 9,
    "axes.labelsize": 9.5, "xtick.labelsize": 9, "ytick.labelsize": 9,
    "axes.spines.top": False, "axes.spines.right": False,
    "grid.color": RULE, "grid.linewidth": 0.7, "grid.linestyle": "-",
    "axes.axisbelow": True, "legend.frameon": False, "legend.fontsize": 9,
    "xtick.major.size": 3, "ytick.major.size": 3,
})

def finish(ax, title, note=None, source=None, grid="y"):
    """Title block, one recessive grid, and a source line. Every chart ends here."""
    ax.set_title(title, pad=24 if note else 9)
    if note:                                      # subtitle, tucked under the title
        ax.text(0, 1.02, note, transform=ax.transAxes, color=MUTED, fontsize=9, va="bottom")
    if grid:
        ax.grid(axis=grid, alpha=1)
        ax.grid(axis="x" if grid == "y" else "y", visible=False)
    if source:
        ax.figure.text(0.005, -0.01, source, color=MUTED, fontsize=7.5, ha="left", va="top")
    sns.despine(ax=ax)

## 1 · A bar, for comparing two things

*Women's Pockets are Inferior* (2018). Someone measured the pockets in 80 pairs of jeans.

In [ ]:
jeans = pudding("pockets/measurements.csv")
print(len(jeans), "pairs,", jeans["brand"].nunique(), "brands")
jeans[["brand", "style", "menWomen", "maxHeightFront", "maxWidthFront"]].head()

In [ ]:
depth = jeans.groupby("menWomen")["maxHeightFront"].mean()
print(depth.round(1))

fig, ax = plt.subplots(figsize=(4.6, 3.4))
ax.bar(["men's", "women's"], [depth["men"], depth["women"]], color=[TEAL, RUST], width=0.5)
for i, v in enumerate([depth["men"], depth["women"]]):
    ax.text(i, v + 0.5, f"{v:.1f} cm", ha="center", color=INK, fontsize=9.5)
ax.set_ylabel("front pocket depth (cm)")
ax.set_ylim(0, 26)
finish(ax, "Women's jeans have shallower pockets",
       source="The Pudding, Women's Pockets are Inferior (2018)")
plt.tight_layout(); plt.show()

Two numbers, two bars. They start at zero, so twice as tall means twice as deep.

## 2 · Bars hide the spread

Those two bars are 80 pairs of jeans squashed into two numbers. Draw all 80.

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 2.9))
# swarmplot nudges overlapping dots sideways instead of stacking them out of sight.
sns.swarmplot(data=jeans, x="maxHeightFront", y="menWomen", order=["men", "women"],
              hue="menWomen", hue_order=["men", "women"], palette=[TEAL, RUST],
              size=6.5, linewidth=0.8, edgecolor=PAPER, legend=False, ax=ax)
for i, group in enumerate(["men", "women"]):
    mean = jeans.loc[jeans["menWomen"] == group, "maxHeightFront"].mean()
    ax.plot([mean, mean], [i - 0.34, i + 0.34], color=INK, lw=1.4, zorder=4)
ax.annotate("average", (jeans.loc[jeans["menWomen"] == "men", "maxHeightFront"].mean(), -0.34),
            xytext=(0, 5), textcoords="offset points", ha="center", va="bottom",
            color=MUTED, fontsize=8.5)
ax.set_ylim(1.62, -0.72)
ax.set_yticks([0, 1], ["men's", "women's"])
ax.set_ylabel("")
ax.set_xlabel("front pocket depth (cm)")
finish(ax, "Every pair, with the average marked", grid="x",
       source="The Pudding, Women's Pockets are Inferior (2018)")
plt.tight_layout(); plt.show()

shallowest_men = jeans.loc[jeans["menWomen"] == "men", "maxHeightFront"].min()
deeper = (jeans.loc[jeans["menWomen"] == "women", "maxHeightFront"] > shallowest_men).sum()
print(f"{deeper} of 40 women's pairs beat the shallowest men's pair ({shallowest_men} cm)")

The two groups barely touch. One women's pair out of forty beats the shallowest men's pair.

The bar chart was accurate. The dots are more convincing.

## 3 · A line, for change over time

*The Names in Songs* — every first name sung on the Billboard Hot 100, 1958 to 2019.

In [ ]:
songs = pudding("names-in-songs/unique.csv")
songs["year"] = pd.to_numeric(songs["year"], errors="coerce")
people = songs[songs["person"] == True]
people = people[people["year"] < 2019]          # 2019 is a part-year, it would dip for no reason
print(len(people), "name mentions,", int(people["year"].min()), "to", int(people["year"].max()))
people[["artist", "song", "name", "year"]].head()

In [ ]:
per_year = people.groupby("year").size()

fig, ax = plt.subplots(figsize=(7.2, 3.2))
ax.plot(per_year.index, per_year.values, color=RUST, lw=2)
ax.annotate(f"{per_year.iloc[-1]:.0f}", (per_year.index[-1], per_year.iloc[-1]),
            xytext=(6, -2), textcoords="offset points", color=INK, fontsize=9.5)
ax.set_ylabel("names sung")
ax.set_xlim(1956, 2022)
ax.set_ylim(0, None)
finish(ax, "Names in hit songs, per year",
       source="The Pudding, The Names in Songs")
plt.tight_layout(); plt.show()
print("1960:", per_year[1960], " 2018:", per_year[2018])

Ten times as many names as in 1960. Before believing that, check what else changed.

The last year in the file stops mid-year, so the code drops it. A half-year plotted beside full
ones looks like a crash.

## 4 · Divide by whatever grew anyway

The dataset has more songs in it every year too. So of course it has more names.

In [ ]:
counted = pd.DataFrame({"names": people.groupby("year").size(),
                        "songs": people.groupby("year")["song"].nunique()})
counted["per_song"] = counted["names"] / counted["songs"]

fig, (a, b) = plt.subplots(1, 2, figsize=(9.2, 3.2), sharex=True)
a.plot(counted.index, counted["names"], color=RUST, lw=2)
a.set_ylim(0, None)
finish(a, "Names, counted")
b.plot(counted.index, counted["per_song"], color=TEAL, lw=2)
b.set_ylim(0, 2.6)
finish(b, "Names per song", source="The Pudding, The Names in Songs")
plt.tight_layout(); plt.show()

print(counted.loc[[1960, 1980, 2000, 2018]].round(2).to_string())

The left chart rises tenfold. The right one goes from about 1.5 to about 2.2. Most of the rise
was the dataset getting bigger, not songwriting changing.

Whenever you plot a count, ask what else got bigger over the same period.

## 5 · When the colour is the data

*The Naked Truth* — 625 foundation shades from 36 makeup brands, each with its own hex code.

In [ ]:
shades = pudding("makeup-shades/shades.csv")
print(len(shades), "shades,", shades["brand"].nunique(), "brands")
shades[["brand", "product", "hex", "L"]].head()

In [ ]:
# L is lightness, 0 darkest to 100 lightest. Draw each shade in its own colour.
BRANDS = ["Fenty", "Maybelline", "MAC", "Estée Lauder", "L'Oréal"]
fig, ax = plt.subplots(figsize=(7.6, 3.1))
for i, brand in enumerate(BRANDS):
    rows = shades[shades["brand"] == brand].sort_values("L")
    ax.scatter(rows["L"], [i] * len(rows), c="#" + rows["hex"], s=110,
               edgecolor=PAPER, linewidth=1.2, zorder=3)
ax.set_yticks(range(len(BRANDS)), BRANDS)
ax.set_xlabel("lightness (0 darkest, 100 lightest)")
ax.set_xlim(5, 100)
ax.invert_yaxis()
finish(ax, "Every shade, in its own colour", grid="x",
       source="The Pudding, The Naked Truth")
plt.tight_layout(); plt.show()

The dots are drawn in the colour they describe, so there is no key to look up.

This is the one job colour does well. Everywhere else, a second colour is one more thing the
reader has to decode, so use it only when you need to tell two things apart.

## 6 · The category is a choice

Back to the songs. Which names get sung most?

In [ ]:
top = people["name"].value_counts().head(10)
print(top.to_string())

fig, ax = plt.subplots(figsize=(6.2, 3.4))
ax.barh(top.index[::-1], top.values[::-1], color=GREY, height=0.68)
ax.text(top.iloc[0] - 40, len(top) - 1, f"{top.iloc[0]:,}", va="center", ha="right",
        color=PAPER, fontsize=9.5)
finish(ax, "Most-sung names", grid="x", source="The Pudding, The Names in Songs")
plt.tight_layout(); plt.show()

"Baby" is 23% of the names in this dataset, and it is not a name. Someone decided it counted.

Drop it and the chart answers a different question.

In [ ]:
real = top.drop(["Baby", "Jesus"])

fig, ax = plt.subplots(figsize=(6.2, 3.4))
ax.barh(real.index[::-1], real.values[::-1], color=RUST, height=0.68)
finish(ax, "Most-sung names, without Baby and Jesus", grid="x",
       source="The Pudding, The Names in Songs")
plt.tight_layout(); plt.show()

Neither chart is wrong. They answer different questions, and what separates them is a decision
made before any code ran. Write that decision down where your readers can see it.

## 7 · The title is the finding

Same numbers, three titles. Only one of them tells the reader what to see.

In [ ]:
depth = jeans.groupby("menWomen")["maxHeightFront"].mean()
TITLES = ["Figure 3", "Front pocket depth by gender",
          "Women's jean pockets are 38% shallower"]

fig, axes = plt.subplots(1, 3, figsize=(10.2, 2.9), sharey=True)
for ax, title in zip(axes, TITLES):
    ax.bar(["men's", "women's"], [depth["men"], depth["women"]], color=[TEAL, RUST], width=0.5)
    finish(ax, title)
    ax.title.set_fontsize(10)
plt.tight_layout(); plt.show()

print(f"{100 * (1 - depth['women'] / depth['men']):.0f}% shallower")

Only the third title tells you anything. Write the title last, once you know what the chart
says, and make it a full sentence.

# Part two: harder charts

These are built the same way as the first seven: marks at coordinates, repeated and labelled.
Same two colours, same `finish()`.

## 8 · Small multiples

*Women in Headlines* measured how emotionally loaded the wording of news headlines is, 0 to 1,
for headlines about women against all headlines.

Four countries on one chart means eight lines crossing each other. Give each country its own
panel on identical axes instead, and you compare by looking from panel to panel.

In [ ]:
tone = pudding("women-in-headlines/polarity_comparison_country_time.csv")
MEANS = ["women_polarity_mean", "all_polarity_mean"]

# These columns never legitimately sit below 0.2, so the exact zeros are missing values
# written as 0. Left in, they draw a line plunging to the floor and back.
print("exact zeros:", int((tone[MEANS] == 0).sum().sum()))
tone[MEANS] = tone[MEANS].replace(0, float("nan"))

countries = sorted(tone["country"].unique())
fig, axes = plt.subplots(1, 4, figsize=(11, 3.1), sharey=True, sharex=True)
for ax, country in zip(axes, countries):
    one = tone[tone["country"] == country].sort_values("year")
    ax.plot(one["year"], one["all_polarity_mean"], color=GREY, lw=2)
    ax.plot(one["year"], one["women_polarity_mean"], color=RUST, lw=2)
    ax.set_xticks([2010, 2015, 2020])
    finish(ax, country)
axes[0].set_ylim(0.10, 0.45)                     # low enough that nothing is cut off
axes[0].set_ylabel("loaded wording")

# A legend, because two series should never be told apart by colour memory alone.
handles = [plt.Line2D([], [], color=RUST, lw=2, label="headlines about women"),
           plt.Line2D([], [], color=GREY, lw=2, label="all headlines")]
fig.legend(handles=handles, loc="lower center", ncol=2, bbox_to_anchor=(0.5, -0.06),
           labelcolor=INK)
plt.tight_layout(); plt.show()

Rust sits above grey in all four panels: headlines about women are worded more strongly than
headlines in general, everywhere.

Small multiples work when the axes are identical, the panels are in an order that means
something, and the labelling happens once instead of in every panel. The gaps in the lines are
years with too few headlines to average. Identical axes make those gaps visible rather than
hiding them.

The panels also show two problems a single chart would have buried. South Africa's last point
runs off the top of the axis and rests on a handful of headlines, so cutting it off is a choice
you should state in the caption. And the US grey line drops sharply in 2016 and recovers, which
nothing here explains. Check a number like that before you build anything on it.

## 9 · A bump chart

Ranks over time. One line per name, moving up and down a top ten, so you can see who arrives
and who drops out.

In [ ]:
decades = pudding("names-in-songs/timeless_names.csv")
decades = decades[decades["person"] == True]
table = decades.pivot_table(index="name", columns="decade", values="rank")
years = sorted(table.columns)
print(len(table), "names hold a top-10 place in at least one decade")

fig, ax = plt.subplots(figsize=(8.4, 5.2))
for name, row in table.iterrows():
    seen = [(y, row[y]) for y in years if pd.notna(row[y])]
    lasting = len(seen) >= 4
    ax.plot([p[0] for p in seen], [p[1] for p in seen],
            color=RUST if lasting else GREY, lw=2 if lasting else 1.1,
            marker="o", ms=7 if lasting else 6, mec=PAPER, mew=1.2,
            zorder=3 if lasting else 2)
    ax.annotate(name, seen[-1], xytext=(9, 0), textcoords="offset points", va="center",
                fontsize=8.5, color=INK if lasting else MUTED, zorder=4,
                bbox=dict(facecolor=PAPER, edgecolor="none", pad=1.4, alpha=0.85))

ax.set_xticks(years)
ax.set_yticks(range(1, 11))
ax.invert_yaxis()
ax.set_ylabel("rank")
ax.set_xlim(1955, 2023)
handles = [plt.Line2D([], [], color=RUST, lw=2, label="holds a place for 4 decades or more"),
           plt.Line2D([], [], color=GREY, lw=1.1, label="everyone else")]
ax.legend(handles=handles, loc="lower left", bbox_to_anchor=(0, -0.2), ncol=2,
          labelcolor=INK)
finish(ax, "Most-sung names, top ten by decade", grid=None,
       source="The Pudding, The Names in Songs")
plt.tight_layout(); plt.show()

Rust marks the names that hold a top-ten place for four decades or more. Grey marks everyone
else, which is most of them: the list turns over almost completely.

That turnover is why this chart type earns its space here. If the lines ran flat and parallel,
a table would have said the same thing faster.

## 10 · A dumbbell chart

Two numbers per row, joined by a line. Each row is a makeup brand, and the two ends are its
darkest and lightest foundation. Rows are sorted by how dark the darkest one is.

In [ ]:
span = shades.groupby("brand")["L"].agg(["min", "max", "count"]).sort_values("min")
span = span[span["count"] >= 6]

fig, ax = plt.subplots(figsize=(7.4, 7.2))
for i, (brand, row) in enumerate(span.iterrows()):
    ax.plot([row["min"], row["max"]], [i, i], color=RULE, lw=2.5, zorder=1,
            solid_capstyle="round")
    ax.scatter([row["min"], row["max"]], [i, i], color=["#4A2E1C", "#EBD3BB"],
               s=85, edgecolor=PAPER, linewidth=1.2, zorder=2)

# The shade count rides the tick label, so it is not one more number floating in the plot.
ax.set_yticks(range(len(span)), [f"{b}  ({int(c)})" for b, c in zip(span.index, span["count"])])
ax.set_xlabel("lightness of the darkest and lightest shade")
ax.set_xlim(2, 100)
ax.invert_yaxis()
handles = [plt.Line2D([], [], marker="o", ls="", mfc="#4A2E1C", mec=PAPER, ms=9, label="darkest"),
           plt.Line2D([], [], marker="o", ls="", mfc="#EBD3BB", mec=PAPER, ms=9, label="lightest")]
ax.legend(handles=handles, loc="lower left", labelcolor=INK)
finish(ax, "How far down each brand goes", note="brand (shades sold)", grid="x",
       source="The Pudding, The Naked Truth")
plt.tight_layout(); plt.show()

The number in brackets after each brand is how many shades it sells. A long line built from six
shades is a thinner offer than a long line built from fifty, and without that number the two
look the same.

Sorting is what makes this readable. In alphabetical order the same rows say nothing.

## 11 · A heatmap

A heatmap suits data with two labels on every number, here a year and a month. Colour carries
the number itself. This is every US birth from 1985 to 2015.

Watch out for one thing in this file: `stateBirths` is repeated on every county row of that
state, so summing it counts each birth dozens of times. Sum `countyBirths` instead.

In [ ]:
import calendar

births = pudding("births/allBirthData.csv")
grid = births.groupby(["Year", "Month"])["countyBirths"].sum().unstack()

# February is short. Without this the chart is mostly a picture of month lengths.
days = pd.DataFrame({m: [calendar.monthrange(y, m)[1] for y in grid.index] for m in grid.columns},
                    index=grid.index)
grid = grid / days

fig, ax = plt.subplots(figsize=(7.2, 6.2))
im = ax.imshow(grid, aspect="auto", cmap=ROAST,
               extent=[0.5, 12.5, grid.index.max() + 0.5, grid.index.min() - 0.5])
ax.set_xticks(range(1, 13), [calendar.month_abbr[m] for m in range(1, 13)], fontsize=8.5)
ax.tick_params(length=0)
for spine in ax.spines.values():
    spine.set_visible(False)
bar = fig.colorbar(im, shrink=0.72, aspect=28, pad=0.03)
bar.set_label("births per day", color=MUTED, fontsize=9)
bar.outline.set_visible(False)
bar.ax.tick_params(length=0, labelsize=8.5, colors=MUTED)
ax.set_title("US births per day")
ax.figure.text(0.005, -0.01, "The Pudding, births data", color=MUTED, fontsize=7.5,
               ha="left", va="top")
plt.tight_layout(); plt.show()

print(grid.mean().round(0).astype(int).to_string())

A bright band runs through late summer, brightest in September, and the whole picture dims
after 2007. Neither shows up in a line of yearly totals.

The colour scale is part of the argument, because a heatmap can only be read against its colour
bar. Use one hue running pale to dark, like the ramp set at the top of this notebook, or a
built-in like `viridis`. Avoid `jet` and `rainbow`: they put sharp colour boundaries where the
data changes smoothly, and they lose their order in greyscale or for a colour-blind reader.

A heatmap also shows where to look rather than how large a difference is. September runs about
11% above January, which these colours make look bigger than it is.

## 12 · A scatter that says no

*The Hype Machine* tracked basketball players ranked in the top 100 coming out of high school,
and what they were worth once they reached the NBA.

If the ranking predicted anything, this chart would slope downwards.

In [ ]:
players = pudding("hype/players.csv").dropna(subset=["rank", "nba_mean_wa"])
print(len(players), "recruits who made the NBA")

fig, ax = plt.subplots(figsize=(7.6, 4.6))
ax.scatter(players["rank"], players["nba_mean_wa"], color=TEAL, alpha=0.5, s=42,
           edgecolor=PAPER, linewidth=0.8, zorder=3)
ax.axhline(0, color=RULE, lw=1, zorder=1)

for _, row in players.nlargest(4, "nba_mean_wa").iterrows():
    ax.annotate(row["name"], (row["rank"], row["nba_mean_wa"]), xytext=(9, -3),
                textcoords="offset points", fontsize=9, color=INK, zorder=4)

r = players["rank"].corr(players["nba_mean_wa"])
ax.set_xlabel("high-school recruit rank (1 is the most hyped)")
ax.set_ylabel("average wins added in the NBA")
ax.set_xlim(-3, 112)
finish(ax, "Recruit rank barely predicts anything", note=f"r = {r:.2f}",
       source="The Pudding, The Hype Machine")
plt.tight_layout(); plt.show()

It does not slope. The top-ranked recruit is LeBron James, and the second-best career here
belongs to the player ranked ninety-fifth.

Be careful how far you take that. Everyone on this chart reached the NBA, so every highly
ranked player who never made it is missing from it. And a flat cloud tells you about these 267
players, not about scouting in general.

A chart that shows no relationship is still a finding. Report it.

## Your turn

The Pudding's [data repo](https://github.com/the-pudding/data) has about forty datasets. Pick
one and make two charts from it: the obvious one, and one that complicates it.

```python
df = pudding("dress-codes/banned_items.csv")     # or vogue/models.csv, boybands/bands.csv
df.head()
```

Check four things before you show anyone a chart:

1. Does the bar start at zero? If not, say why.
2. Have you hidden a spread inside an average?
3. Did the count go up, or did the dataset?
4. Does the title say what you found?